In [0]:
from pyspark.sql import functions as F


In [0]:
##function to check schema, null values, distinct, numeric statistic 
from pyspark.sql import DataFrame
def profile_data(df: DataFrame) -> None:
    ###print schema
    print("Schema")
    df.printSchema()

    ###null values
    print("Null values")
    df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()

    ###distinct values
    print("Distinct values")
    df.select([
        F.countDistinct(c).alias(c)
        for c in df.columns
    ]).show()

    ###5. Numeric statistics
    print("Numeric statistics")
    df.summary().show()




In [0]:
print("ERP profiling")
##cust_az12 profile
print("cust_az12")
df_cust_az12 = spark.read.table("medallionsales.bronze.cust_az12")
profile_data(df_cust_az12)
##loc_a101 profile
print("loc_a101")
df_loc_a101 = spark.read.table("medallionsales.bronze.loc_a101")
profile_data(df_loc_a101)
##px_cat_g1v2 profile
print("px_cat_g1v2")
df_px_cat_g1v2 = spark.read.table("medallionsales.bronze.px_cat_g1v2")
profile_data(df_px_cat_g1v2)

print("CRM profiling")
##cust_info profile
print("cust_info")
df_cust_info = spark.read.table("medallionsales.bronze.cust_info")
profile_data(df_cust_info)
##prd_info profile
print("prd_info")
df_prd_info = spark.read.table("medallionsales.bronze.prd_info")
profile_data(df_prd_info)
##sales_details profile
print("sales_details")
df_sales_details = spark.read.table("medallionsales.bronze.sales_details")
profile_data(df_sales_details)



ERP profiling
cust_az12
Schema
root
 |-- CID: string (nullable = true)
 |-- BDATE: date (nullable = true)
 |-- GEN: string (nullable = true)
 |-- _ingest_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

Null values
+---+-----+----+-----------------+------------+
|CID|BDATE| GEN|_ingest_timestamp|_source_file|
+---+-----+----+-----------------+------------+
|  0|    0|1472|                0|           0|
+---+-----+----+-----------------+------------+

Distinct values
+-----+-----+---+-----------------+------------+
|  CID|BDATE|GEN|_ingest_timestamp|_source_file|
+-----+-----+---+-----------------+------------+
|18484| 6152|  8|                1|           1|
+-----+-----+---+-----------------+------------+

Numeric statistics
+-------+-------------+-----+--------------------+
|summary|          CID|  GEN|        _source_file|
+-------+-------------+-----+--------------------+
|  count|        18484|17012|               18484|
|   mean|         NULL| 

In [0]:
##customer id/key duplication
df_cust_info.groupBy("cst_id").count().filter("count > 1").show()
df_cust_info.groupBy("cst_key").count().filter("count > 1").show()


+------+-----+
|cst_id|count|
+------+-----+
| 29433|    2|
|  NULL|    4|
| 29449|    2|
| 29466|    3|
| 29473|    2|
| 29483|    2|
+------+-----+

+----------+-----+
|   cst_key|count|
+----------+-----+
|AW00029433|    2|
|AW00029449|    2|
|AW00029466|    3|
|AW00029473|    2|
|AW00029483|    2|
+----------+-----+



In [0]:
##Noted: records with same id at least one of them has full name(fname, lname)
duplicate_ids = (
    df_cust_info
    .groupBy("cst_id")
    .count()
    .filter("count > 1")
    .select("cst_id")
)

df_cust_info.join(
    duplicate_ids,
    on="cst_id",
    how="inner"
).show()

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|   _ingest_timestamp|        _source_file|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+--------------------+
| 29433|AW00029433|         NULL|        NULL|                 M|       M|     2026-01-25|2026-09-28 05:09:...|abfss://sink-sys@...|
| 29433|AW00029433|       Thomas|        King|                 M|       M|     2026-01-27|2026-09-28 05:09:...|abfss://sink-sys@...|
| 29449|AW00029449|         NULL|        Chen|                 S|    NULL|     2026-01-25|2026-09-28 05:09:...|abfss://sink-sys@...|
| 29449|AW00029449|        Laura|        Chen|                 S|       F|     2026-01-26|2026-09-28 05:09:...|abfss://sink-sys@...|
| 29466|AW00029466|         NULL|        NULL|              NULL|    

In [0]:
#validation
# duplicates = (
#     df_cust_info
#     .groupBy("cst_id")
#     .agg(
#         F.count("*").alias("cnt"),
#         F.sum(
#             F.when(
#                 F.col("cst_firstname").isNull() |
#                 F.col("cst_lastname").isNull(),
#                 1
#             ).otherwise(0)
#         ).alias("null_name_rows")
#     )
#     .filter(F.col("cnt") > 1)
# )

# duplicates.select(
#     F.count("*").alias("duplicate_ids"),
#     F.sum(
#         F.when(F.col("null_name_rows") > 0, 1).otherwise(0)
#     ).alias("duplicates_with_null_name")
# ).show()

duplicates = df_cust_info.groupBy("cst_id")\
    .agg(
        F.count("*").alias("cnt"),

        F.sum(
            F.when(
                F.col("cst_firstname").isNull() | F.col("cst_lastname").isNull(),1
            ).otherwise(0)
        ).alias("null_name_rows")
    ).filter(F.col("cnt")>1)


duplicates.select(
    F.count("*").alias("diplicates_ids"),
    F.sum(
        F.when(F.col("null_name_rows")>0,1).otherwise(0)
    ).alias("duplicates_with_null_name")
).show()



+--------------+-------------------------+
|diplicates_ids|duplicates_with_null_name|
+--------------+-------------------------+
|             6|                        6|
+--------------+-------------------------+



In [0]:
print("Invalid SCD dates:",
      df_prd_info.filter(F.col("prd_start_dt") > F.col("prd_end_dt")).count())

print("Zero order dates:",
      df_sales_details.filter(F.col("sls_order_dt") == 0).count())

print("Negative sales:",
      df_sales_details.filter(F.col("sls_sales") < 0).count())

print("Negative prices:",
      df_sales_details.filter(F.col("sls_price") < 0).count())

Invalid SCD dates: 200
Zero order dates: 17
Negative sales: 3
Negative prices: 5


In [0]:
##in source system there are some records with invalid date range
##it was swapped (start_dt <---> end_dt)
df_prd_info.select("prd_key", "prd_start_dt", "prd_end_dt") \
    .filter(
        F.col("prd_end_dt").isNotNull() &
        (F.col("prd_start_dt") > F.col("prd_end_dt"))
    ) \
    .show()

+----------------+------------+----------+
|         prd_key|prd_start_dt|prd_end_dt|
+----------------+------------+----------+
| AC-HE-HL-U509-R|  2011-07-01|2007-12-28|
| AC-HE-HL-U509-R|  2012-07-01|2008-12-27|
|   AC-HE-HL-U509|  2011-07-01|2007-12-28|
|   AC-HE-HL-U509|  2012-07-01|2008-12-27|
| CL-SO-SO-B909-M|  2011-07-01|2007-12-28|
| CL-SO-SO-B909-L|  2011-07-01|2007-12-28|
| AC-HE-HL-U509-B|  2011-07-01|2007-12-28|
| AC-HE-HL-U509-B|  2012-07-01|2008-12-27|
|   CL-CA-CA-1098|  2011-07-01|2007-12-28|
|   CL-CA-CA-1098|  2012-07-01|2008-12-27|
| CL-JE-LJ-0192-S|  2011-07-01|2007-12-28|
| CL-JE-LJ-0192-S|  2012-07-01|2008-12-27|
| CL-JE-LJ-0192-M|  2011-07-01|2007-12-28|
| CL-JE-LJ-0192-M|  2012-07-01|2008-12-27|
| CL-JE-LJ-0192-L|  2011-07-01|2007-12-28|
| CL-JE-LJ-0192-L|  2012-07-01|2008-12-27|
| CL-JE-LJ-0192-X|  2011-07-01|2007-12-28|
| CL-JE-LJ-0192-X|  2012-07-01|2008-12-27|
|CO-RF-FR-R92R-62|  2011-07-01|2007-12-28|
|CO-RF-FR-R92R-62|  2012-07-01|2008-12-27|
+----------

In [0]:
##two distinct values swapped
df_prd_info \
    .filter(
        F.col("prd_end_dt").isNotNull() &
        (F.col("prd_start_dt") > F.col("prd_end_dt"))
    ) \
    .groupBy("prd_start_dt", "prd_end_dt") \
    .count() \
    .orderBy("prd_start_dt", "prd_end_dt") \
    .show(5)

+------------+----------+-----+
|prd_start_dt|prd_end_dt|count|
+------------+----------+-----+
|  2011-07-01|2007-12-28|   72|
|  2012-07-01|2008-12-27|  128|
+------------+----------+-----+



In [0]:
##rows with order date == 0 are suspicious
##Noted: the ship date always greater than order date + 7 days
df_sales_details.filter(F.col("sls_order_dt") == 0) \
    .select(
        "sls_ord_num",
        "sls_order_dt",
        "sls_ship_dt",
        "sls_due_dt"
    ).show()

+-----------+------------+-----------+----------+
|sls_ord_num|sls_order_dt|sls_ship_dt|sls_due_dt|
+-----------+------------+-----------+----------+
|    SO64338|           0|   20130823|  20130828|
|    SO64339|           0|   20130823|  20130828|
|    SO64339|           0|   20130823|  20130828|
|    SO64340|           0|   20130823|  20130828|
|    SO64340|           0|   20130823|  20130828|
|    SO64377|           0|   20130824|  20130829|
|    SO64377|           0|   20130824|  20130829|
|    SO64379|           0|   20130824|  20130829|
|    SO64381|           0|   20130824|  20130829|
|    SO64623|           0|   20130827|  20130901|
|    SO64624|           0|   20130827|  20130901|
|    SO64625|           0|   20130827|  20130901|
|    SO64743|           0|   20130829|  20130903|
|    SO64744|           0|   20130829|  20130903|
|    SO65337|           0|   20130905|  20130910|
|    SO65338|           0|   20130905|  20130910|
|    SO65340|           0|   20130905|  20130910|


In [0]:
##negative in sales and price
df_sales_details \
    .filter(
        (F.col("sls_sales") < 0) |
        (F.col("sls_price") < 0)
    ) \
    .select(
        "sls_ord_num",
        "sls_prd_key",
        "sls_quantity",
        "sls_price",
        "sls_sales"
    ) \
    .show()

+-----------+-----------+------------+---------+---------+
|sls_ord_num|sls_prd_key|sls_quantity|sls_price|sls_sales|
+-----------+-----------+------------+---------+---------+
|    SO57804| BK-M38S-40|           1|     -769|      769|
|    SO57916|    TI-M602|           1|      -30|       30|
|    SO58326|    FE-6654|           1|      -22|       22|
|    SO58623| BK-R79Y-44|           1|    -1701|     1701|
|    SO58818|    TI-R092|           1|      -21|       21|
|    SO61570|    CA-1098|           1|        9|      -18|
|    SO69066|  SJ-0194-L|           1|       54|      -54|
|    SO69215|    TI-M823|           1|       35|      -35|
+-----------+-----------+------------+---------+---------+



In [0]:
##track historical changes for one sample
df_prd_info \
    .filter(F.col("prd_key") == "AC-HE-HL-U509-R") \
    .select(
        "prd_id",
        "prd_key",
        "prd_start_dt",
        "prd_end_dt"
    ) \
    .orderBy("prd_start_dt") \
    .show()

+------+---------------+------------+----------+
|prd_id|        prd_key|prd_start_dt|prd_end_dt|
+------+---------------+------------+----------+
|   212|AC-HE-HL-U509-R|  2011-07-01|2007-12-28|
|   213|AC-HE-HL-U509-R|  2012-07-01|2008-12-27|
|   214|AC-HE-HL-U509-R|  2013-07-01|      NULL|
+------+---------------+------------+----------+



In [0]:
##sales amount is not equal to quantity * price
df_sales_details \
    .filter(
        F.col("sls_sales") !=
        F.col("sls_quantity") * F.col("sls_price")
    ) \
    .select(
        "sls_ord_num",
        "sls_prd_key",
        "sls_quantity",
        "sls_price",
        "sls_sales"
    ) \
    .show(20)

+-----------+-----------+------------+---------+---------+
|sls_ord_num|sls_prd_key|sls_quantity|sls_price|sls_sales|
+-----------+-----------+------------+---------+---------+
|    SO57804| BK-M38S-40|           1|     -769|      769|
|    SO57916|    TI-M602|           1|      -30|       30|
|    SO58326|    FE-6654|           1|      -22|       22|
|    SO58335|    TI-M823|           2|       35|       35|
|    SO58623| BK-R79Y-44|           1|    -1701|     1701|
|    SO58818|    TI-R092|           1|      -21|       21|
|    SO61548|  LJ-0192-L|           2|       50|       50|
|    SO61570|    CA-1098|           1|        9|      -18|
|    SO61636|    BC-M005|           1|       10|        0|
|    SO61879|  LJ-0192-X|           2|       50|       50|
|    SO67707|    TT-M928|           2|        5|        5|
|    SO67707|  VE-C304-S|           4|       64|       64|
|    SO67708|    TI-M267|           3|       25|       25|
|    SO67993|    PK-7098|           1|        2|       4